# 07 - Train Per-Crop Disease Classifier (EfficientNet-B2)

Stage 7: Train one disease classifier per crop (EfficientNet-B2).

Reads disease_dataset/<crop>/{train,val,test}/<disease>/ produced by
06_prepare_disease_dataset.py. Trains ONE crop at a time (pass --crop),
or all five sequentially if you leave --crop out -- useful since your
GPU will be busy a while and you may want to run crops one at a time
between other work.

Saves each crop's model + label list separately:
    models/disease_<crop>.pth
    models/disease_<crop>_labels.json

Set CROP_TO_TRAIN below and run -- set it to None to train all 5 crops
back-to-back in one run (only do this if your GPU has time to spare;
each crop takes as long as 02_train_crop_identifier.py did per epoch).

Install deps (same as 02_train_crop_identifier.py):
    pip install torch torchvision timm albumentations scikit-learn --break-system-packages

## Imports & Configuration

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.metrics import classification_report
import cv2

DATA_ROOT = Path("disease_dataset")
MODELS_DIR = Path("models")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROPS = ["Cotton", "Groundnut", "Pepper Bell", "Potato", "Tomato"]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(12, 24), hole_width_range=(12, 24), p=0.3),  # simulates leaf-spot occlusion
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

g:\Crop Identification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\Crop Identification\venv\Lib\site-packages\albumentations\check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()


## `AlbumentationsImageFolder`

In [2]:
class AlbumentationsImageFolder(Dataset):
    def __init__(self, root, transform):
        self.base = ImageFolder(root)
        self.transform = transform
        self.classes = self.base.classes

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        path, label = self.base.samples[idx]
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, label

## `build_weighted_sampler`

In [3]:
def build_weighted_sampler(dataset):
    targets = [label for _, label in dataset.base.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

## `train_one_crop`

In [4]:
from tqdm.auto import tqdm

def train_one_crop(crop_name):
    crop_slug = crop_name.replace(" ", "_")
    train_dir = DATA_ROOT / crop_slug / "train"
    val_dir = DATA_ROOT / crop_slug / "val"

    if not train_dir.exists():
        print(f"Skipping {crop_name}: {train_dir} not found (did 06 run for this crop?)")
        return

    print(f"\n{'=' * 20} Training disease classifier: {crop_name} {'=' * 20}")

    train_ds = AlbumentationsImageFolder(train_dir, train_tf)
    val_ds = AlbumentationsImageFolder(val_dir, eval_tf)
    num_classes = len(train_ds.classes)
    print(f"Classes ({num_classes}): {train_ds.classes}")

    if num_classes < 2:
        print(f"Skipping {crop_name}: fewer than 2 surviving classes after pruning in step 06.")
        return

    sampler = build_weighted_sampler(train_ds)
    
    # Set num_workers=0 to prevent Windows hanging/freezing issues
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = timm.create_model("efficientnet_b2", pretrained=True, num_classes=num_classes)
    model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience, patience_counter = 5, 0
    model_path = MODELS_DIR / f"disease_{crop_slug}.pth"
    labels_path = MODELS_DIR / f"disease_{crop_slug}_labels.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        # 1. Training Loop with Live Progress Bar
        train_pbar = tqdm(train_loader, desc=f"  [{crop_name}] Epoch {epoch+1}/{EPOCHS} Train", leave=False)
        for images, labels in train_pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        scheduler.step()
        train_loss = running_loss / len(train_ds)

        # 2. Validation Loop with Live Progress Bar
        model.eval()
        correct, total = 0, 0
        all_preds, all_labels = [], []
        
        val_pbar = tqdm(val_loader, desc=f"  [{crop_name}] Epoch {epoch+1}/{EPOCHS} Val  ", leave=False)
        with torch.no_grad():
            for images, labels in val_pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_acc = correct / total

        print(f"[{crop_name}] Epoch {epoch+1}/{EPOCHS} - train_loss: {train_loss:.4f} - val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), model_path)
            with open(labels_path, "w") as f:
                json.dump(train_ds.classes, f)
            print(f"  -> saved new best model (val_acc={val_acc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[{crop_name}] Early stopping triggered.")
                break

    print(f"[{crop_name}] Best val_acc: {best_val_acc:.4f}. Model saved to {model_path}")

    print(f"\n[{crop_name}] Val-set classification report (last epoch):")
    print(classification_report(all_labels, all_preds, target_names=train_ds.classes, digits=3, zero_division=0))

## Run

In [10]:
CROP_TO_TRAIN = "Tomato"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Tomato ====================
Classes (8): ['Bacterial Spot', 'Early Blight', 'Healthy', 'Late Blight', 'Mold Leaf', 'Mosaic Virus', 'Septoria', 'Yellow Curl Virus']


[Tomato] Epoch 1/25 - train_loss: 0.9632 - val_acc: 0.8233
  -> saved new best model (val_acc=0.8233)


[Tomato] Epoch 2/25 - train_loss: 0.3319 - val_acc: 0.8750
  -> saved new best model (val_acc=0.8750)


[Tomato] Epoch 3/25 - train_loss: 0.2022 - val_acc: 0.8879
  -> saved new best model (val_acc=0.8879)


[Tomato] Epoch 4/25 - train_loss: 0.1557 - val_acc: 0.8933
  -> saved new best model (val_acc=0.8933)


...

[Tomato] Epoch 18/25 - train_loss: 0.0180 - val_acc: 0.9278


[Tomato] Epoch 19/25 - train_loss: 0.0129 - val_acc: 0.9300


[Tomato] Epoch 20/25 - train_loss: 0.0138 - val_acc: 0.9267


[Tomato] Epoch 21/25 - train_loss: 0.0125 - val_acc: 0.9246


[Tomato] Epoch 22/25 - train_loss: 0.0133 - val_acc: 0.9300
[Tomato] Early stopping triggered.
[Tomato] Best val_acc: 0.9300. Model saved to models\disease_Tomato.pth

[Tomato] Val-set classification report (last epoch):
                   precision    recall  f1-score   support

   Bacterial Spot      0.983     0.956     0.969       182
     Early Blight      0.924     0.945     0.935       219
          Healthy      0.957     0.985     0.971       205
      Late Blight      0.939     0.939     0.939       165
        Mold Leaf      0.872     0.895     0.883        76
     Mosaic Virus      0.688     0.786     0.733        14
         Septoria      0.838     0.689     0.756        45
Yellow Curl Virus      0.750     0.682     0.714        22

         accuracy                          0.930       928
        macro avg      0.869     0.860     0.863       928
     weighted avg      0.930     0.930     0.929       928



In [11]:
CROP_TO_TRAIN = "Cotton"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Cotton ====================
Classes (8): ['Alternaria Leaf Spot', 'Bacterial Blight', 'Curl Virus', 'Fusarium Wilt', 'Healthy', 'Powdery Mildew', 'Target Spot', 'Verticillium Wilt']


[Cotton] Epoch 1/25 - train_loss: 0.9961 - val_acc: 0.8412
  -> saved new best model (val_acc=0.8412)


[Cotton] Epoch 2/25 - train_loss: 0.2409 - val_acc: 0.8736
  -> saved new best model (val_acc=0.8736)


[Cotton] Epoch 3/25 - train_loss: 0.1893 - val_acc: 0.9134
  -> saved new best model (val_acc=0.9134)


[Cotton] Epoch 4/25 - train_loss: 0.1100 - val_acc: 0.9278
  -> saved new best model (val_acc=0.9278)


...

[Cotton] Epoch 19/25 - train_loss: 0.0177 - val_acc: 0.9711


[Cotton] Epoch 20/25 - train_loss: 0.0060 - val_acc: 0.9783


[Cotton] Epoch 21/25 - train_loss: 0.0207 - val_acc: 0.9783


[Cotton] Epoch 22/25 - train_loss: 0.0103 - val_acc: 0.9783


[Cotton] Epoch 23/25 - train_loss: 0.0027 - val_acc: 0.9783
[Cotton] Early stopping triggered.
[Cotton] Best val_acc: 0.9783. Model saved to models\disease_Cotton.pth

[Cotton] Val-set classification report (last epoch):
                      precision    recall  f1-score   support

Alternaria Leaf Spot      1.000     1.000     1.000        26
    Bacterial Blight      0.956     0.977     0.966        44
          Curl Virus      0.923     1.000     0.960        12
       Fusarium Wilt      0.989     1.000     0.994        88
             Healthy      1.000     0.946     0.972        56
      Powdery Mildew      1.000     1.000     1.000         5
         Target Spot      0.667     0.667     0.667         6
   Verticillium Wilt      1.000     1.000     1.000        40

            accuracy                          0.978       277
           macro avg      0.942     0.949     0.945       277
        weighted avg      0.979     0.978     0.978       277



In [5]:
CROP_TO_TRAIN = "Groundnut"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Groundnut ====================
Classes (6): ['Alternaria Leaf Spot', 'Healthy', 'Leaf Spot', 'Nutrition Deficiency', 'Rosette', 'Rust']


[Groundnut] Epoch 1/25 - train_loss: 0.4961 - val_acc: 0.8329
  -> saved new best model (val_acc=0.8329)


[Groundnut] Epoch 2/25 - train_loss: 0.1842 - val_acc: 0.8602
  -> saved new best model (val_acc=0.8602)


[Groundnut] Epoch 3/25 - train_loss: 0.1648 - val_acc: 0.8818
  -> saved new best model (val_acc=0.8818)


[Groundnut] Epoch 4/25 - train_loss: 0.1169 - val_acc: 0.8545


...

[Groundnut] Epoch 18/25 - train_loss: 0.0191 - val_acc: 0.9409


[Groundnut] Epoch 19/25 - train_loss: 0.0177 - val_acc: 0.9409


[Groundnut] Epoch 20/25 - train_loss: 0.0134 - val_acc: 0.9409


[Groundnut] Epoch 21/25 - train_loss: 0.0125 - val_acc: 0.9352


[Groundnut] Epoch 22/25 - train_loss: 0.0110 - val_acc: 0.9409
[Groundnut] Early stopping triggered.
[Groundnut] Best val_acc: 0.9409. Model saved to models\disease_Groundnut.pth

[Groundnut] Val-set classification report (last epoch):
                      precision    recall  f1-score   support

Alternaria Leaf Spot      0.966     0.982     0.974        57
             Healthy      0.912     0.956     0.934       228
           Leaf Spot      0.951     0.925     0.938       294
Nutrition Deficiency      0.980     0.960     0.970        50
             Rosette      1.000     0.929     0.963        14
                Rust      0.939     0.902     0.920        51

            accuracy                          0.941       694
           macro avg      0.958     0.942     0.950       694
        weighted avg      0.942     0.941     0.941       694



In [6]:
CROP_TO_TRAIN = "Pepper Bell"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Pepper Bell ====================
Classes (7): ['Bacterial Spot', 'Cercospora Leaf Spot', 'Edema', 'Healthy', 'Leaf Curl', 'Nutrient Deficiency', 'Powdery Mildew']


[Pepper Bell] Epoch 1/25 - train_loss: 0.2078 - val_acc: 0.9919
  -> saved new best model (val_acc=0.9919)


[Pepper Bell] Epoch 2/25 - train_loss: 0.0475 - val_acc: 0.9848


[Pepper Bell] Epoch 3/25 - train_loss: 0.0391 - val_acc: 0.9928
  -> saved new best model (val_acc=0.9928)


[Pepper Bell] Epoch 4/25 - train_loss: 0.0318 - val_acc: 0.9928


...

[Pepper Bell] Epoch 8/25 - train_loss: 0.0053 - val_acc: 0.9919


[Pepper Bell] Epoch 9/25 - train_loss: 0.0046 - val_acc: 0.9937


[Pepper Bell] Epoch 10/25 - train_loss: 0.0117 - val_acc: 0.9928


[Pepper Bell] Epoch 11/25 - train_loss: 0.0092 - val_acc: 0.9946


[Pepper Bell] Epoch 12/25 - train_loss: 0.0106 - val_acc: 0.9919
[Pepper Bell] Early stopping triggered.
[Pepper Bell] Best val_acc: 0.9946. Model saved to models\disease_Pepper_Bell.pth

[Pepper Bell] Val-set classification report (last epoch):
                      precision    recall  f1-score   support

      Bacterial Spot      0.994     1.000     0.997       535
Cercospora Leaf Spot      1.000     0.991     0.995       211
               Edema      0.625     0.833     0.714         6
             Healthy      0.991     0.991     0.991       225
           Leaf Curl      1.000     0.981     0.990        52
 Nutrient Deficiency      0.983     0.983     0.983        58
      Powdery Mildew      1.000     0.929     0.963        28

            accuracy                          0.992      1115
           macro avg      0.942     0.958     0.948      1115
        weighted avg      0.993     0.992     0.992      1115



In [ ]:
CROP_TO_TRAIN = "Potato"  # set to one of CROPS, or None to train all 5

crops_to_run = [CROP_TO_TRAIN] if CROP_TO_TRAIN else CROPS
for crop in crops_to_run:
    train_one_crop(crop)


==================== Training disease classifier: Potato ====================
Classes (8): ['Bacteria', 'Early Blight', 'Fungi', 'Healthy', 'Late Blight', 'Nematode', 'Pest', 'Virus']


[Potato] Epoch 1/25 - train_loss: 0.7345 - val_acc: 0.9001
  -> saved new best model (val_acc=0.9001)


[Potato] Epoch 2/25 - train_loss: 0.2495 - val_acc: 0.9183
  -> saved new best model (val_acc=0.9183)


[Potato] Epoch 3/25 - train_loss: 0.1717 - val_acc: 0.9244
  -> saved new best model (val_acc=0.9244)


[Potato] Epoch 4/25 - train_loss: 0.1245 - val_acc: 0.9288
  -> saved new best model (val_acc=0.9288)


...

[Potato] Epoch 19/25 - train_loss: 0.0108 - val_acc: 0.9496


[Potato] Epoch 20/25 - train_loss: 0.0065 - val_acc: 0.9496


[Potato] Epoch 21/25 - train_loss: 0.0041 - val_acc: 0.9505


[Potato] Epoch 22/25 - train_loss: 0.0044 - val_acc: 0.9513
[Potato] Early stopping triggered.
[Potato] Best val_acc: 0.9566. Model saved to models\disease_Potato.pth

[Potato] Val-set classification report (last epoch):
              precision    recall  f1-score   support

    Bacteria      0.944     0.988     0.966        85
Early Blight      1.000     1.000     1.000       265
       Fungi      0.879     0.862     0.870       109
     Healthy      0.952     0.978     0.965       224
 Late Blight      0.990     0.990     0.990       290
    Nematode      0.833     0.909     0.870        11
        Pest      0.864     0.787     0.824        89
       Virus      0.857     0.846     0.852        78

    accuracy                          0.951      1151
   macro avg      0.915     0.920     0.917      1151
weighted avg      0.951     0.951     0.951      1151



: 